# TTC Subway Delay Data

Analysis of TTC subway delay records (2018-2025+), sourced from the City of Toronto Open Data Portal.

## 1. Setup

In [ ]:
import pandas as pd
import requests
from io import BytesIO
import re
import matplotlib.pyplot as plt

## 2. Load Delay Codes Reference Data

Fetch the `ttc-subway-delay-codes` resource, which documents every delay code and its description. Split into `codes_df1` (subway/SUB codes) and `codes_df2` (SRT codes).

In [ ]:
import requests

# Toronto Open Data is stored in a CKAN instance. It's APIs are documented here:
# https://docs.ckan.org/en/latest/api/

# To hit our API, you'll be making requests to:
base_url = "https://ckan0.cf.opendata.inter.prod-toronto.ca"

# Datasets are called "packages". Each package can contain many "resources"
# To retrieve the metadata for this package and its resources, use the package name in this page's URL:
url = base_url + "/api/3/action/package_show"
params = { "id": "ttc-subway-delay-data"}
package = requests.get(url, params = params).json()

# To get resource data:
for idx, resource in enumerate(package["result"]["resources"]):

       # To get metadata for non datastore_active resources:
       if not resource["datastore_active"]:
           url = base_url + "/api/3/action/resource_show?id=" + resource["id"]
           resource_metadata = requests.get(url).json()
           print(resource_metadata)
           # From here, you can use the "url" attribute to download this file


In [ ]:
import requests
from io import BytesIO

# pick the specific resource you want, e.g. by name, rather than relying on the last loop iteration
target = next(
    r for r in package["result"]["resources"]
    if r["name"] == "ttc-subway-delay-codes"
)

resource_metadata = requests.get(base_url + "/api/3/action/resource_show?id=" + target["id"]).json()
file_url = resource_metadata["result"]["url"]

resp = requests.get(file_url, headers={"User-Agent": "Mozilla/5.0"})
resp.raise_for_status()

print(resp.headers.get("Content-Type"))  # sanity check it's really an xlsx

df_codes = pd.read_excel(BytesIO(resp.content), skiprows=1, engine="openpyxl")


In [ ]:
df_codes

In [ ]:
codes_df1 = df_codes[['SUB RMENU CODE', 'CODE DESCRIPTION']].copy()
codes_df2 = df_codes[['SRT RMENU CODE', 'CODE DESCRIPTION']].copy()

In [ ]:
codes_df1

In [ ]:
codes_df2

## 3. Load 2018 Data (Initial Single-Year Exploration)

Before building the full multi-year pipeline, load a single year (2018) first to understand the data's structure.

In [ ]:
import requests
from io import BytesIO

# pick the specific resource you want, e.g. by name, rather than relying on the last loop iteration
target = next(
    r for r in package["result"]["resources"]
    if r["name"] == "ttc-subway-delay-data-2018"
)

resource_metadata = requests.get(base_url + "/api/3/action/resource_show?id=" + target["id"]).json()
file_url = resource_metadata["result"]["url"]

resp = requests.get(file_url, headers={"User-Agent": "Mozilla/5.0"})
resp.raise_for_status()

print(resp.headers.get("Content-Type"))  # sanity check it's really an xlsx

df_2018 = pd.read_excel(BytesIO(resp.content),  engine="openpyxl") #skiprows=1,


In [ ]:
df_2018

## 4. ETL: Consolidate All Years (2018 - 2025+)

Loop through each year's resource (2018-2024 as XLSX, 2025+ as CSV) and concatenate into a single dataframe, `df_all`.

In [ ]:
import requests
from io import BytesIO

years = range(2018, 2025)  # 2018 a 2024 (xlsx)
all_dfs = []

for year in years:
    resource_name = f"ttc-subway-delay-data-{year}"
    
    target = next(
        (r for r in package["result"]["resources"] if r["name"] == resource_name),
        None
    )
    
    if target is None:
        print(f"No se encontró recurso para {year}, revisar nombre")
        continue
    
    resource_metadata = requests.get(
        base_url + "/api/3/action/resource_show?id=" + target["id"]
    ).json()
    file_url = resource_metadata["result"]["url"]
    
    resp = requests.get(file_url, headers={"User-Agent": "Mozilla/5.0"})
    resp.raise_for_status()
    
    df_year = pd.read_excel(BytesIO(resp.content), engine="openpyxl")
    df_year["Year"] = year
    
    all_dfs.append(df_year)
    print(f"{year}: {df_year.shape[0]} filas cargadas")

# --- Caso especial: datos desde 2025 en adelante, vienen como CSV ---
target_2025 = next(
    (r for r in package["result"]["resources"] if r["name"] == "TTC Subway Delay Data since 2025"),
    None
)

if target_2025 is not None:
    resource_metadata = requests.get(
        base_url + "/api/3/action/resource_show?id=" + target_2025["id"]
    ).json()
    file_url = resource_metadata["result"]["url"]
    
    resp = requests.get(file_url, headers={"User-Agent": "Mozilla/5.0"})
    resp.raise_for_status()
    
    df_2025 = pd.read_csv(BytesIO(resp.content))
    df_2025["Year"] = 2025  # o extraer el año real de la columna Date si abarca varios años
    
    all_dfs.append(df_2025)
    print(f"2025+: {df_2025.shape[0]} filas cargadas")
else:
    print("No se encontró el recurso CSV de 2025")

df_all = pd.concat(all_dfs, ignore_index=True)
df_all.shape

## 5. Data Quality & Homogeneity Checks

Before analysis, check for structural breaks or inconsistencies across years: does 2026 data leak into the 'since 2025' resource? Are row counts, schemas, and zero-delay ratios consistent year to year? What does the official readme document about the fields?

In [ ]:
df_all['Date'] = pd.to_datetime(df_all['Date'])
df_all[df_all['Date'].dt.year == 2026]

In [ ]:
for d in all_dfs:
    yr = d['Year'].iloc[0]
    zero_pct = (d['Min Delay'] == 0).mean() * 100
    print(yr, '| rows:', d.shape[0], '| columns:', d.columns.tolist(), '| %zero delay:', round(zero_pct,1))

In [ ]:
target_readme = next(r for r in package["result"]["resources"] if r["name"] == "ttc-subway-delay-data-readme")
resource_metadata = requests.get(base_url + "/api/3/action/resource_show?id=" + target_readme["id"]).json()
file_url = resource_metadata["result"]["url"]
resp = requests.get(file_url, headers={"User-Agent": "Mozilla/5.0"})
resp.raise_for_status()
xls = pd.ExcelFile(BytesIO(resp.content), engine="openpyxl")
print(xls.sheet_names)
for s in xls.sheet_names:
    print('---', s, '---')
    print(xls.parse(s).head(20))

## 6. Line Column Exploration (Draft - Not Yet Applied)

The `Line` column contains many inconsistent variants (e.g. `YU/BD`, `YUS`, even bus route numbers entered by mistake). This section explores the raw values and drafts a standardization function. **Not yet applied to `df_all`/`df_recent`** - to be refined and applied in a future step.

In [ ]:
df_all['Line'].value_counts()

In [ ]:
import re

def standardize_line(val):
    if pd.isna(val):
        return val
    v = str(val).upper().strip()

    # Junk / non-line values -> Unknown
    junk_patterns = ['TRACK LEVEL', 'STN', '999']
    if any(p in v for p in junk_patterns) or re.match(r'^\d{2,3}\s', v):
        return 'UNKNOWN'  # catches bus route numbers like "506 CARLTON"

    # Normalize separators so combos are consistent
    v_norm = re.sub(r'[\s\-&/]+', '/', v)  # turn -, &, spaces, slashes into a single "/"
    v_norm = v_norm.replace('LINES', '').replace('LINE', '').strip('/')

    # Map old naming (YUS) to current (YU)
    v_norm = v_norm.replace('YUS', 'YU')

    # Known single-line names spelled out
    if 'BLOOR' in v_norm or v_norm in ('BD', 'B/D'):
        return 'BD'
    if v_norm in ('SHEP', 'SHP'):
        return 'SHP'
    if v_norm == 'EC' or 'ECLRT' in v_norm:
        return 'EC'
    if v_norm == 'FWLRT':
        return 'FWLRT'
    if v_norm == 'SRT':
        return 'SRT'
    if v_norm == 'YU':
        return 'YU'

    # Multi-line combos -> sort parts alphabetically so "YU/BD" and "BD/YU" match
    parts = sorted(set(p for p in v_norm.split('/') if p))
    if parts:
        return '/'.join(parts)

    return 'UNKNOWN'

df_all['Line_clean'] = df_all['Line'].apply(standardize_line)
df_all['Line_clean'].value_counts()

## 7. Filter to Comparable Period (2022 - 2025)

2018-2021 has a much lower reporting volume than 2022+, and Line 3 (SRT) permanently closed in 2023. To keep comparisons fair, restrict analysis to 2022-2025, deriving the real year from `Date` rather than the hardcoded `Year` column.

In [ ]:
df_all['Date'] = pd.to_datetime(df_all['Date'])
df_all['RealYear'] = df_all['Date'].dt.year

df_recent = df_all[(df_all['RealYear'] >= 2022) & (df_all['RealYear'] <= 2025)].copy()
df_recent['RealYear'].value_counts().sort_index()

## 8. Data Exploration (2022-2025)

Before ranking causes, look at the overall shape of `df_recent`: how delay minutes are distributed, how records break down by line, and how they break down by day of week.

### Summary Statistics

In [ ]:
df_recent[['Min Delay', 'Min Gap']].describe()

### Distribution of Delay Minutes (Delayed Records Only)

In [ ]:
delay_only = df_recent[df_recent['Min Delay'] > 0]['Min Delay']
delay_only.describe()

In [ ]:
fig, ax = plt.subplots(figsize=(10,5))
ax.hist(delay_only, bins=50, range=(0, delay_only.quantile(0.95)), color='steelblue', edgecolor='white')
ax.set_xlabel('Min Delay (minutes)')
ax.set_ylabel('Number of Records')
ax.set_title('Distribution of Delay Minutes (records with Min Delay > 0, 95th percentile cap)')
plt.tight_layout()
plt.show()

### Records by Line

In [ ]:
df_recent['Line'].value_counts()

### Records by Day of Week

In [ ]:
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
day_counts = df_recent['Day'].value_counts().reindex(day_order)
day_counts

In [ ]:
fig, ax = plt.subplots(figsize=(8,5))
ax.bar(day_counts.index, day_counts.values, color='steelblue')
ax.set_ylabel('Number of Records')
ax.set_title('Delay Records by Day of Week (2022-2025)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 9. Goal 1: Main Causes of Delay - Pareto Analysis

Rank delay causes by total delay minutes (not just frequency), and check what share of total delay time the top causes represent.

### Aggregation

In [ ]:
pareto = df_recent.groupby('Code')['Min Delay'].sum().sort_values(ascending=False).reset_index()
pareto.columns = ['Code', 'TotalDelayMinutes']
pareto['CumPct'] = pareto['TotalDelayMinutes'].cumsum() / pareto['TotalDelayMinutes'].sum() * 100

top20 = pareto.head(20)
top20

### Visualization

In [ ]:
fig, ax1 = plt.subplots(figsize=(12,6))
ax1.bar(top20['Code'], top20['TotalDelayMinutes'], color='steelblue')
ax1.set_ylabel('Total Delay Minutes')
ax1.set_xticklabels(top20['Code'], rotation=90)

ax2 = ax1.twinx()
ax2.plot(top20['Code'], top20['CumPct'], color='red', marker='o')
ax2.set_ylabel('Cumulative %')
ax2.axhline(80, color='gray', linestyle='--')

plt.title('Pareto: Top 20 Delay Causes by Total Delay Minutes (2022-2025)')
plt.tight_layout()
plt.show()

### Code Definitions (Top 3 Causes by Total Delay Minutes)

In [ ]:
codes_df1[codes_df1['SUB RMENU CODE'].isin(['SUDP','SUUT','SUO'])]

## 10. Goal 1b: Category-Level Pareto (Grouped by Code Prefix)

`codes_df1` has no explicit category column, only individual `SUB RMENU CODE` values and descriptions. Looking at the codes, they consistently follow a one-letter-prefix pattern (`S*`, `M*`, `P*`, `T*`, `E*`). Grouping by the first letter of the code gives a coarser view. Note this is a grouping derived from the observed prefix pattern, not an official TTC-published category label.

### Aggregation

In [ ]:
df_recent['CodeCategory'] = df_recent['Code'].str[0]
category_pareto = df_recent.groupby('CodeCategory')['Min Delay'].sum().sort_values(ascending=False).reset_index()
category_pareto.columns = ['CodeCategory', 'TotalDelayMinutes']
category_pareto['CumPct'] = category_pareto['TotalDelayMinutes'].cumsum() / category_pareto['TotalDelayMinutes'].sum() * 100
category_pareto

### Visualization

In [ ]:
fig, ax1 = plt.subplots(figsize=(8,5))
ax1.bar(category_pareto['CodeCategory'], category_pareto['TotalDelayMinutes'], color='steelblue')
ax1.set_ylabel('Total Delay Minutes')
ax1.set_xlabel('Code Category (first letter of Code)')

ax2 = ax1.twinx()
ax2.plot(category_pareto['CodeCategory'], category_pareto['CumPct'], color='red', marker='o')
ax2.set_ylabel('Cumulative %')
ax2.axhline(80, color='gray', linestyle='--')
ax2.set_ylim(0,105)

plt.title('Pareto: Delay Causes Grouped by Code Category (2022-2025)')
plt.tight_layout()
plt.show()

### What's in Each Category?

In [ ]:
codes_df1['Category'] = codes_df1['SUB RMENU CODE'].str[0]
codes_df1.groupby('Category')['CODE DESCRIPTION'].apply(lambda d: list(d)[:5])